In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/isramoussaoui/coding/backend/.env
/kaggle/input/datasets/isramoussaoui/coding/backend/test_new_features.py
/kaggle/input/datasets/isramoussaoui/coding/backend/video_processing.py
/kaggle/input/datasets/isramoussaoui/coding/backend/.gitignore
/kaggle/input/datasets/isramoussaoui/coding/backend/run_backend.ipynb
/kaggle/input/datasets/isramoussaoui/coding/backend/main.py
/kaggle/input/datasets/isramoussaoui/coding/backend/PR_DESCRIPTION.md
/kaggle/input/datasets/isramoussaoui/coding/backend/prompt2dict.py
/kaggle/input/datasets/isramoussaoui/coding/backend/qa_agent.py
/kaggle/input/datasets/isramoussaoui/coding/backend/graph_store.py
/kaggle/input/datasets/isramoussaoui/coding/backend/visual_summary.py
/kaggle/input/datasets/isramoussaoui/coding/backend/summary.py
/kaggle/input/datasets/isramoussaoui/coding/backend/prompts.py
/kaggle/input/datasets/isramoussaoui/coding/backend/requirements.txt
/kaggle/input/datasets/isramoussaoui/coding/backend/llm_call.py
/kaggle/

# Backend runner: graph store + `/ask` (Neo4j-backed) — clean run/test notebook

This notebook replaces the ad-hoc cells from your working notebook with **one linear path**:
install → cookies → Neo4j (via Kaggle Secret, not a hardcoded password) → start the API
→ exercise every retrieval path in `/ask` → verify what actually landed in the graph.

Before running: add two **Kaggle Secrets** (Add-ons → Secrets):
- `NGROK_AUTHTOKEN` (already required by your original notebook)
- `NEO4J_PASSWORD` (pick any string — this run's DB is thrown away when the session ends anyway)


In [1]:
BACKEND_DIR = "/kaggle/input/datasets/isramoussaoui/coding/backend"

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/isramoussaoui/coding/backend/.env
/kaggle/input/datasets/isramoussaoui/coding/backend/test_new_features.py
/kaggle/input/datasets/isramoussaoui/coding/backend/video_processing.py
/kaggle/input/datasets/isramoussaoui/coding/backend/.gitignore
/kaggle/input/datasets/isramoussaoui/coding/backend/run_backend.ipynb
/kaggle/input/datasets/isramoussaoui/coding/backend/main.py
/kaggle/input/datasets/isramoussaoui/coding/backend/PR_DESCRIPTION.md
/kaggle/input/datasets/isramoussaoui/coding/backend/prompt2dict.py
/kaggle/input/datasets/isramoussaoui/coding/backend/qa_agent.py
/kaggle/input/datasets/isramoussaoui/coding/backend/graph_store.py
/kaggle/input/datasets/isramoussaoui/coding/backend/visual_summary.py
/kaggle/input/datasets/isramoussaoui/coding/backend/summary.py
/kaggle/input/datasets/isramoussaoui/coding/backend/prompts.py
/kaggle/input/datasets/isramoussaoui/coding/backend/requirements.txt
/kaggle/input/datasets/isramoussaoui/coding/backend/llm_call.py
/kaggle/

## 1. Cookies + working-dir env vars (writable copies only — input dir stays read-only)

In [2]:
import os, shutil

SOURCE_COOKIES = f"{BACKEND_DIR}/cookies.txt"
WORKING_COOKIES = "/kaggle/working/cookies.txt"

if os.path.exists(SOURCE_COOKIES):
    shutil.copy2(SOURCE_COOKIES, WORKING_COOKIES)
    print(f"Cookies copied to: {WORKING_COOKIES}")
else:
    print(f"WARNING: cookies file not found at {SOURCE_COOKIES} (yt-dlp will fall back to no-cookie mode)")

os.environ["VIDEO_COOKIES_FILE"] = WORKING_COOKIES
os.environ["VIDEO_WORK_DIR"] = "/kaggle/working"


Cookies copied to: /kaggle/working/cookies.txt


## 2. Install dependencies (one cell, includes everything the code actually imports)

In [8]:
!pip install -q -r /kaggle/input/datasets/isramoussaoui/coding/backend/requirements.txt
!pip install -q -U groq neo4j pyngrok nest_asyncio scikit-learn easyocr
!python -m spacy download en_core_web_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 3.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 12.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 53.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.8 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.6 MB/s eta 0:00:0000:0100:01
✔ Download and installat

## 3. Neo4j — download once, start once

Password comes from the `NEO4J_PASSWORD` Kaggle Secret, not typed in plaintext.
Safe to re-run: it checks whether Neo4j is already listening before doing anything.


In [3]:
import os
os.environ["VIDEO_WORK_DIR"] = "/kaggle/working/video_data"

os.makedirs("/kaggle/working/video_data", exist_ok=True)

print(os.environ["VIDEO_WORK_DIR"])

/kaggle/working/video_data


In [4]:
import os, socket, subprocess, time
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["NEO4J_PASSWORD"] = user_secrets.get_secret("NEO4J_PASSWORD")

NEO4J_HOME = "/kaggle/working/neo4j-community-5.24.0"

def neo4j_running():
    try:
        with socket.create_connection(("localhost", 7687), timeout=1):
            return True
    except OSError:
        return False

if not neo4j_running():
    if not os.path.exists(NEO4J_HOME):
        print("Downloading Neo4j...")
        subprocess.run(
            ["wget", "-q", "https://dist.neo4j.org/neo4j-community-5.24.0-unix.tar.gz",
             "-O", "/kaggle/working/neo4j.tar.gz"],
            check=True,
        )
        subprocess.run(["tar", "-xzf", "/kaggle/working/neo4j.tar.gz", "-C", "/kaggle/working/"], check=True)
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)

    # Set the password once (fails harmlessly if already set from a prior run in this session)
    subprocess.run(
        [f"{NEO4J_HOME}/bin/neo4j-admin", "dbms", "set-initial-password", os.environ["NEO4J_PASSWORD"]],
    )

    print("Starting Neo4j...")
    log_file = open("/kaggle/working/neo4j.log", "a")
    subprocess.Popen(
        [f"{NEO4J_HOME}/bin/neo4j", "console"],
        stdout=log_file, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True,
    )

    for i in range(60):
        if neo4j_running():
            print(f"Neo4j is up after {i}s")
            break
        time.sleep(1)
    else:
        raise RuntimeError("Neo4j did not start in time -- check /kaggle/working/neo4j.log")
else:
    print("Neo4j already running on bolt://localhost:7687")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20+8-1~22.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20+8-1~22.04) over (17.0.18+8-1~22.04.1) ...
Preparing to unpack .../openjdk-17-jre-headless_17.0.20+8-1~22.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20+8-1~22.04) over (17.0.18+8-1~22.04.1) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20+8-1~22.04) ...
Installing new version of config file /etc/java-17-openjdk/security/java.security ...
Installing new version of config file /etc/java-17-openjdk/security/public_suffix_list.dat ...
Setting up openjdk-17-jdk-headless:amd64 (17.0.20+8-1~22.04) ...
Changed password for user 'neo4j'. IMPORTANT: this change will only take effect if performed before the database is started for the first time.
Starting Neo4j...
Neo4j is up after 9s


In [5]:
# Sanity check the connection directly (separate from the app's own driver)
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()
print("Neo4j connection OK")


Neo4j connection OK


## 4. Start the API — ONE path only

`main.py`'s own `if __name__ == "__main__"` block already starts ngrok *and* uvicorn together.
Running it as a script (`python main.py`) in the background and reading its own log for the
printed tunnel URL avoids the double-start / duplicate-tunnel problem in the old notebook.


In [10]:
import os
import signal

try:
    api_process.terminate()
    api_process.wait(timeout=5)
except:
    pass

In [6]:
!ps aux | grep -E "main.py|ngrok" | grep -v grep

In [22]:
!pkill -f "main.py"
!pkill -f "ngrok"

In [7]:
import subprocess
import time
import re
import os

os.chdir(BACKEND_DIR)

log_path = "/kaggle/working/backend.log"

# Start backend
log_file = open(log_path, "w")

api_process = subprocess.Popen(
    ["python", "-u", "main.py"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    cwd=BACKEND_DIR,
)

public_url = None

for i in range(60):
    time.sleep(1)

    with open(log_path, "r") as f:
        text = f.read()

    # Look for the URL printed by main.py
    m = re.search(
        r"Backend URL:\s*(https?://\S+)",
        text
    )

    if m:
        public_url = m.group(1).rstrip()
        print(f"Backend URL: {public_url}")
        break

    if "NGROK FAILED" in text:
        print("========== NGROK ERROR ==========")
        print(text)
        print("=================================")
        raise RuntimeError("ngrok failed to start")

else:
    print("========== BACKEND LOG ==========")
    print(text)
    print("=================================")
    raise RuntimeError(
        "Server did not report a URL in time"
    )

# Keep the log file open while the backend runs
log_file.flush()

Backend URL: https://pendant-lavender-underpay.ngrok-free.dev


In [8]:
import requests
# confirm the app itself (not just ngrok) is actually serving
for i in range(30):
    try:
        r = requests.get(f"{public_url}/", timeout=3)
        if r.ok:
            print(r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("API never came up -- check /kaggle/working/backend.log")


{'message': 'Video Summarizer API is running.'}


## 5. Exercise the pipeline + the three new pieces

Run `/summarize` and `/getcode` on one or two videos first -- this is what populates the
graph and the vector index. Then hit `/ask` with questions designed to trip each routing
branch in `qa_agent.py`:

1. relational ("which videos mention X") → Cypher traversal
2. semantic ("what did they say about Y") → vector search over Segment chunks
3. hybrid → semantic hit whose topic is also a graph Entity
4. fallback → question about a video/topic with nothing indexed yet


In [9]:
import shutil

SOURCE = "/kaggle/input/datasets/isramoussaoui/coding/backend"
DEST = "/kaggle/working/backend"

shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)

print("Copied to:", DEST)

Copied to: /kaggle/working/backend


In [10]:
VIDEO_URL = "https://www.youtube.com/watch?v=iWS9ogMPOI0"

resp = requests.post(
    f"{public_url}/summarize",
    json={"vid_url": VIDEO_URL},
    timeout=600
)

print("Status:", resp.status_code)
print("Response:")
print(resp.text)

Status: 200
Response:
{"goal":"The video demonstrates how to install FastAPI, build a simple API app, and explore its features such as routing, Pydantic models, and interactive documentation.","global_understanding":"FastAPI is introduced as an async-first Python web framework designed for building high‑performance APIs. The tutorial covers installation with Uvicorn, creating routes, handling data with Pydantic models, managing errors with HTTP exceptions, and using built‑in Swagger and ReDoc interfaces for testing and documentation. It also compares FastAPI to Flask and Django and hints at next steps like database integration and deployment.","steps":["Step 1: Install FastAPI and Uvicorn, create a `main.py` with a root route that returns “hello world”, and run the server using `uvicorn main:app --reload`.","Step 2: Add CRUD routes for a to‑do list: maintain an in‑memory list, create POST and GET endpoints, implement error handling with `HTTPException`, and model request/response data 

In [13]:
print(open("/kaggle/working/backend.log").read()[-10000:])

or` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
----------------Downloading the video------------
DEBUG cookies_file = '/kaggle/working/cookies.txt'
DEBUG ydl_opts keys = ['outtmpl', 'format', 'merge_output_format', 'retries', 'fragment_retries', 'format_sort', 'prefer_free_formats', 'extractor_args', 'http_headers', 'cookiefile']
[youtube] Extracting URL: https://www.youtube.com/watch?v=iWS9ogMPOI0
[youtube] iWS9ogMPOI0: Downloading webpage
[youtube] iWS9ogMPOI0: Downloading initial data API JSON
[youtube] iWS9ogMPOI0: Downloading visionos player API JSON
ERROR: [youtube] iWS9ogMPOI0: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how

In [11]:
import video_processing

print(video_processing.__file__)
print(video_processing.WORK_DIR)

/kaggle/input/datasets/isramoussaoui/coding/backend/video_processing.py
/kaggle/working/video_data


In [14]:
resp = requests.post(
    f"{public_url}/getcode",
    json={"vid_url": VIDEO_URL},
    timeout=600
)

print("Status:", resp.status_code)
print("Headers:", resp.headers)
print("Response:")
print(resp.text[:5000])

Status: 503
Headers: {'Connection': 'close', 'Content-Type': 'text/html', 'Ngrok-Error-Code': 'ERR_NGROK_3004', 'Referrer-Policy': 'no-referrer', 'Date': 'Thu, 10 Sep 2026 14:02:56 GMT', 'Transfer-Encoding': 'chunked'}
Response:
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" a

In [16]:
def ask(question, vid_url=None):
    payload = {"question": question}
    if vid_url:
        payload["vid_url"] = vid_url
    r = requests.post(f"{public_url}/ask", json=payload, timeout=120)
    print("Q:", question)
    print("status:", r.status_code)
    body = r.json()
    print("A:", body.get("answer"))
    print("sources:", len(body.get("sources", [])))
    print("-" * 60)
    return body

# Guard against 'summary' not existing yet (e.g. /summarize cell wasn't run,
# or was run but errored before assignment) or having no entities extracted.
entities = summary.get("entities", []) if "summary" in dir() and summary else []
relational_q = f"which videos mention {entities[0][0]}" if entities else "which videos mention Python"

# 1) relational
r1 = ask(relational_q)

# 2) semantic -- vector search over transcript/code chunks, no exact entity match needed
r2 = ask("what did they explain about the main topic", vid_url=VIDEO_URL)

# 3) hybrid -- check body['sources'] for a trailing {'related_entities': [...]} entry
print("hybrid grounding present:", any("related_entities" in s for s in r2.get("sources", []) if isinstance(s, dict)))

# 4) fallback -- ask about a video that was never processed
r4 = ask("what is covered in this video", vid_url="https://www.youtube.com/watch?v=doesnotexist000")

Q: which videos mention Python
status: 200
A: No stored videos currently mention 'Python'.
sources: 0
------------------------------------------------------------
Q: what did they explain about the main topic
status: 200
A: They gave a quick‑start walkthrough of FastAPI: how to install it with Uvicorn, set up a basic `main.py`, create routes (e.g., for a to‑do list), and run the async server. The video highlighted FastAPI’s built‑in, interactive Swagger UI for testing APIs, its automatic OpenAPI JSON export, and the fact it’s async by default—plus a brief comparison with older frameworks like Flask.
sources: 5
------------------------------------------------------------
hybrid grounding present: False
Q: what is covered in this video
status: 200
A: This video hasn't been processed/stored yet.
sources: 0
------------------------------------------------------------


## 6. Verify directly in the graph (bypasses the API entirely)

In [17]:
with driver.session() as session:
    print("Segments per video:")
    for r in session.run("MATCH (v:Video)-[:HAS_SEGMENT]->(s:Segment) RETURN v.url AS url, count(s) AS chunks"):
        print(" ", r["url"], "->", r["chunks"], "chunks")

    print("Entities per video:")
    for r in session.run("MATCH (v:Video)-[:MENTIONS]->(e:Entity) RETURN v.url AS url, count(e) AS entities"):
        print(" ", r["url"], "->", r["entities"], "entities")

    print("Code snippets:")
    for r in session.run("MATCH (v:Video)-[:HAS_CODE]->(c:CodeSnippet) RETURN v.url AS url, count(c) AS n"):
        print(" ", r["url"], "->", r["n"], "snippet(s)")


Segments per video:
  https://www.youtube.com/watch?v=iWS9ogMPOI0 -> 29 chunks
Entities per video:
  https://www.youtube.com/watch?v=iWS9ogMPOI0 -> 3 entities
Code snippets:


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: unknown relationship type. The relationship type `HAS_CODE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=19, offset=18>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'reltype': 'HAS_CODE'}, '_severity': 'WARNING', '_position': {'offset': 18, 'line': 1, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (v:Video)-[:HAS_CODE]->(c:CodeSnippet) RETURN v.url AS url, count(c) AS n'
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: unknown label. The label `CodeSnippet` does not exist. Verify that the spelling is correct.', po

## 7. Shut down cleanly

Run this before re-running from the top in the same session, otherwise you'll get
"port already in use" on 8001.


In [ ]:
api_process.terminate()
try:
    api_process.wait(timeout=5)
except Exception:
    api_process.kill()

from pyngrok import ngrok
ngrok.kill()
print("Stopped.")
